# components.scrollbar

> Custom scrollbar component with proportional thumb for position indication.

In [ ]:
#| default_exp components.scrollbar

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fasthtml.common import Div

from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionConfig, VirtualCollectionState
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

from cjm_fasthtml_virtual_scrollbar.core.models import ScrollbarConfig, ScrollbarState, ScrollbarIds
from cjm_fasthtml_virtual_scrollbar.components.scrollbar import (
    render_scrollbar as _sb_render_scrollbar,
    render_scrollbar_thumb as _sb_render_scrollbar_thumb,
)

In [ ]:
#| export
def _map_to_scrollbar(
    state: VirtualCollectionState,
    config: VirtualCollectionConfig,
    ids: VirtualCollectionHtmlIds,
):  # (ScrollbarState, ScrollbarConfig, ScrollbarIds)
    """Map VC types to scrollbar lib types."""
    sb_state = ScrollbarState(
        position=state.window_start,
        visible_count=state.visible_rows,
        total_items=state.total_items,
    )
    sb_config = ScrollbarConfig(
        prefix=config.prefix,
        show_scrollbar=config.show_scrollbar,
        min_thumb_height=config.min_thumb_height,
    )
    sb_ids = ScrollbarIds(prefix=ids.prefix)
    return sb_state, sb_config, sb_ids


def render_scrollbar_thumb(
    state: VirtualCollectionState,       # Collection state
    config: VirtualCollectionConfig,      # Collection config
    ids: VirtualCollectionHtmlIds,        # HTML IDs
    track_height: float = 600.0,          # Track height for min thumb calculation
    oob: bool = False,                    # Whether to include hx-swap-oob
) -> Div:  # Thumb element
    """Render the scrollbar thumb at the correct position."""
    sb_state, sb_config, sb_ids = _map_to_scrollbar(state, config, ids)
    return _sb_render_scrollbar_thumb(sb_state, sb_config, sb_ids, track_height, oob)

In [ ]:
#| export
def render_scrollbar(
    state: VirtualCollectionState,       # Collection state
    config: VirtualCollectionConfig,      # Collection config
    ids: VirtualCollectionHtmlIds,        # HTML IDs
) -> Div:  # Complete scrollbar (track + thumb)
    """Render the custom scrollbar with track and proportional thumb."""
    sb_state, sb_config, sb_ids = _map_to_scrollbar(state, config, ids)
    return _sb_render_scrollbar(sb_state, sb_config, sb_ids)

## Tests

In [ ]:
from fasthtml.common import to_xml
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionConfig, VirtualCollectionState
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

config = VirtualCollectionConfig(prefix="t", min_thumb_height=24)
ids = VirtualCollectionHtmlIds(prefix="t")

In [ ]:
# Test scrollbar at start
state = VirtualCollectionState(total_items=500, visible_rows=15, window_start=0)
sb = render_scrollbar(state, config, ids)
html = to_xml(sb)
assert 'id="t-scrollbar-track"' in html
assert 'id="t-scrollbar-thumb"' in html
assert 'top: 0.00%' in html
assert 'cursor-grab' in html
assert 'data-visible-count="15"' in html  # Now uses scrollbar lib naming
print("Scrollbar at start test passed")

In [ ]:
# Test scrollbar at end
state = VirtualCollectionState(total_items=500, visible_rows=15, window_start=485)
sb = render_scrollbar(state, config, ids)
html = to_xml(sb)
# Thumb should be near the bottom
assert 'id="t-scrollbar-thumb"' in html
print("Scrollbar at end test passed")

Scrollbar at end test passed


In [ ]:
# Test scrollbar hidden when all items visible — uses display:hidden, not bare div
state = VirtualCollectionState(total_items=10, visible_rows=15, window_start=0)
sb = render_scrollbar(state, config, ids)
html = to_xml(sb)
assert 'id="t-scrollbar-track"' in html
assert 'id="t-scrollbar-thumb"' not in html  # No thumb when all visible
assert 'hidden' in html  # display:hidden class prevents layout flash
print("Scrollbar hidden test passed")

In [ ]:
# Test OOB thumb
state = VirtualCollectionState(total_items=500, visible_rows=15, window_start=100)
thumb = render_scrollbar_thumb(state, config, ids, oob=True)
html = to_xml(thumb)
assert 'hx-swap-oob="outerHTML"' in html
assert 'id="t-scrollbar-thumb"' in html
print("OOB thumb test passed")

OOB thumb test passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()